# Analysis of Eniac Orders

## Cleanup

### Issues

#### General
- [x] inconsistent presence of `sku` and `order_id` &rarr; use intersections between tables

#### Orderlines
- [x] `product_id` contains no information &rarr; dropped
- [x] `unit_price` is type `str` &rarr; first `.` removed, then successfully converted
- [x] `date` no date type &rarr; `datetime64[s]`
- [ ] is `product_quantity` of 999 realistic?
- [ ] huge variation of `unit_price` between orders of the same product

#### Products
- [x] `price` is type `str` &rarr; replace values like 6.555.999 and fill missing values with highest values from orderlines / `unit_price`
- [x] missing values: `desc`: 7, `price` 46, `type` 50 &rarr; after filtering: 6 missing `desc` filled with `name`, `price` filled with max `unit_price` from orderlines, 14 missing `type` rows dropped
- [x] `promo_price` column not usable &rarr; dropped
- [x] duplicate `sku` numbers &rarr; remove duplicates, if one misses a value, use value from other row with same `sku` so that `sku` is ultimately unique
- [ ] `sku` only semi-standardised (e.g. `-A` suffix)
- [x] `type` column damaged (e.g. `1,44E+11`) &rarr; Contverted to respective int, best we can do
- [x] 8746 duplicate rows &rarr; duplicates removed

#### Orders
- [x] missing values: `total_paid`: 5 &rarr; sum of all unit_prices per order
- [x] `date_created` no date type &rarr; `datetime64[s]`
- [ ] Many orders created in the same second
- [ ] 5 most expensive orders have similar totals
- [ ] orders with `total_pay == 0`
- [x] probably not all order states interesting for analysis &rarr; Filtering in the beginning by global variable

#### Brands
- [ ] more unique short than long names

### Set up the DataFrames and Global Variables

#### Imports

In [1]:
import os
import re
import pandas as pd

#### Global Variables

In [157]:
DATA_DIR = "data"
DESIRED_COLS = {
    "brands": [("short", "str"), ("long", "str")],
    "orderlines": [
        ("id", "int"),
        ("id_order", "int"),
        ("product_quantity", "int"),
        ("sku", "str"),
        ("unit_price", "float"),
        ("date", "datetime64[s]"),
    ],
    "orders": [
        ("order_id", "int"),
        ("created_date", "datetime64[s]"),
        ("total_paid", "float"),
        ("state", "str"),
    ],
    "products": [
        ("sku", "str"),
        ("name", "str"),
        ("desc", "str"),
        ("price", "float"),
#        ("promo_price", "float"),
        ("in_stock", "boolean"),
        ("type", "Int64"),
    ],
}
# Select from ['Cancelled', 'Completed', 'Pending', 'Shopping Basket', 'Place Order']
DESIRED_STATES = ["Completed", "Pending", "Place Order"]

In [158]:
def print_rows(dfs):
    for key, df in dfs.items():
        print(f"{key}:")
        print(f"{'='*(len(key)+1)}")
        print(df.shape[0])
        print()

#### Read DataFrames

In [159]:
dfs = {
    f: pd.read_csv(
        os.path.join(DATA_DIR, f + ".csv"), usecols=[i[0] for i in DESIRED_COLS[f]]
    )
    for f in DESIRED_COLS
}

In [160]:
dfs["orders"].groupby("state").count()

,order_id,created_date,total_paid
state,,,
Cancelled,7233,7233,7233
Completed,46605,46605,46605
Pending,14379,14379,14374
Place Order,40883,40883,40883
Shopping Basket,117809,117809,117809


### Remove duplicates, make ids coherent between tables, fix straightforward data types

#### Remove Duplicate `sku` in products

In [161]:
dfs["products"] = dfs["products"].groupby("sku", as_index=False).first()

#### Select desired order states

In [162]:
mask_states = dfs["orders"].loc[:, "state"].isin(DESIRED_STATES)
dfs["orders"] = dfs["orders"].loc[mask_states]

#### Ensure `product_id` and `order_id` are consistent across tables

In [163]:
skus = set(dfs["products"].loc[:, "sku"]) & set(dfs["orderlines"].loc[:, "sku"])
dfs["products"] = dfs["products"].loc[dfs["products"].loc[:, "sku"].isin(skus)]
dfs["orderlines"] = dfs["orderlines"].loc[dfs["orderlines"].loc[:, "sku"].isin(skus)]
print(len(skus))

7850


In [164]:
order_ids = set(dfs["orders"].loc[:, "order_id"].unique()) & set(dfs["orderlines"].loc[:, "id_order"])
dfs["orders"] = dfs["orders"].loc[dfs["orders"].loc[:, "order_id"].isin(order_ids)]
dfs["orderlines"] = dfs["orderlines"].loc[dfs["orderlines"].loc[:, "id_order"].isin(order_ids)]
print(len(order_ids))

89233


Do this again after the `order_id` filtering, because the latter removed some `sku`s from the orderlines table.

In [165]:
skus = set(dfs["products"].loc[:, "sku"]) & set(dfs["orderlines"].loc[:, "sku"])
dfs["products"] = dfs["products"].loc[dfs["products"].loc[:, "sku"].isin(skus)]
dfs["orderlines"] = dfs["orderlines"].loc[dfs["orderlines"].loc[:, "sku"].isin(skus)]
print(len(skus))

6899


#### Fix products / type values

In [166]:
m = dfs["products"].loc[:, "type"].str.contains("E")
dfs["products"].loc[m, "type"] = (
    dfs["products"].loc[m, "type"].apply(lambda s: str(int(float(s.replace(",", ".")))))
)

#### Set Desired Types, where straightforward

In [167]:
types_to_fix = []
values_missing = []
for table, df in dfs.items():
    df = df.drop_duplicates()
    for col, t in DESIRED_COLS[table]:
        # Check for missing values
        nulls = df.loc[:, col].isnull().sum()
        if nulls > 0:
            values_missing.append((table, col, nulls))

        # Check for correct type, try to fix it
        if df.loc[:, col].dtype != t:
            try:
                if t == "datetime":
                    df[col] = pd.to_datetime(df[col], errors="coerce")
                else:
                    df[col] = df[col].astype(t)
            except (ValueError, TypeError):
                types_to_fix.append((table, col))
    dfs[table] = df

print("Types to fix:")
for table, col in types_to_fix:
    print(f"{table}: {col}")

print("\nMissing values:")
for table, col, nulls in values_missing:
    print(f"{table} / {col}: {nulls}")

Types to fix:
orderlines: unit_price
products: price

Missing values:
orders / total_paid: 5
products / desc: 5
products / price: 23
products / type: 13


### Set indices, fill in missing values

#### Set indices

In [168]:
dfs["products"] = dfs["products"].set_index("sku")
dfs["orderlines"] = dfs["orderlines"].set_index("id")
dfs["orders"] = dfs["orders"].set_index("order_id")

#### Fix orderlines unit_price type

In [169]:
def fix_price(df, col_name):
    df.loc[:, col_name] = df.loc[:, col_name].apply(
        lambda s: s[:-7] + s[-6:] if re.match(r"^\d+\.\d{3}\.\d{2}$", s) and pd.notna(s) else s
    )
    return df.astype({col_name: "float"})

In [170]:
dfs["orderlines"] = fix_price(dfs["orderlines"], "unit_price")

#### Fill missing / odd prices using highest price from orderlines

In [171]:
mask_missing_prices = dfs["products"].loc[:, "price"].isnull()
mask_odd_numbers = dfs["products"].loc[:, "price"].apply(lambda s: bool(re.match(r"\d+\.\d{3}\.\d{3}$", s)) if pd.notna(s) else False) # like 6.55.99
skus = dfs["products"].loc[mask_missing_prices | mask_odd_numbers].index
#skus = dfs["products"].loc[mask_missing_prices].index

mask_skus = dfs["orderlines"].loc[:, "sku"].isin(skus)
sku_prices = dfs["orderlines"].loc[mask_skus].groupby("sku").agg(**{"unit_price_max": ("unit_price", "max")})

In [172]:
dfs["products"].loc[sku_prices.index, "price"] = sku_prices.values.astype("str")
dfs["products"] = fix_price(dfs["products"], "price")

#### Add missing description for completness: copy name

In [173]:
mask_desc = dfs["products"].loc[:, "desc"].isnull()
dfs["products"].loc[mask_desc, "desc"] = dfs["products"].loc[:, "name"]

In [174]:
mask_type = dfs["products"].loc[:, "type"].isnull()
dfs["products"] = dfs["products"].loc[~mask_type]

#### Fill missing `total_paid` values in orders

In [175]:
mask_missing = dfs["orders"].loc[:, "total_paid"].isnull()
mask_missing_ol = dfs["orderlines"].loc[:, "id_order"].isin(dfs["orders"].loc[mask_missing].index)
order_total = dfs["orderlines"].loc[mask_missing_ol].groupby("id_order").agg(**{"total_order": ("unit_price", "sum")})
dfs["orders"].loc[order_total.index, "total_paid"] = order_total.values

In [176]:
print_rows(dfs)

brands:
187

orderlines:
114711

orders:
89233

products:
6886



In [177]:
types_to_fix = []
values_missing = []
for table, df in dfs.items():
    for col, t in DESIRED_COLS[table]:
        # Check for missing values
        nulls = df.reset_index().loc[:, col].isnull().sum()
        if nulls > 0:
            values_missing.append((table, col, nulls))

        # Check for correct type, try to fix it
        if df.reset_index().loc[:, col].dtype != t:
            types_to_fix.append((table, col))

print("Types to fix:")
for table, col in types_to_fix:
    print(f"{table}: {col}")

print("\nMissing values:")
for table, col, nulls in values_missing:
    print(f"{table} / {col}: {nulls}")

Types to fix:

Missing values:


### Write csv files

In [179]:
write_csv = False
if write_csv:
    for table, df in dfs.items():
        df.to_csv(os.path.join(DATA_DIR, table + "_clean.csv"))

## Analysis

In [45]:
dfs["orders"].resample("W", on="created_date")

In [44]:
dfs["orders"]

,created_date,total_paid,state
order_id,,,
241423,2017-11-06 13:10:02,136.15,Completed
242832,2017-12-31 17:40:03,15.76,Completed
243330,2017-02-16 10:59:38,84.98,Completed
245275,2017-06-28 11:35:37,149.00,Completed
245595,2017-01-21 12:52:47,112.97,Completed
...,...,...,...
527392,2018-03-14 13:55:11,34.99,Pending
527394,2018-03-14 13:55:15,161.00,Place Order
527396,2018-03-14 13:55:44,141.58,Place Order


#### Compose table containing min / max unit_price, and ratio

In [75]:
p = dfs["products"]
o = dfs["orderlines"]
# NOTE
# APP2203       2539  24.120.043            6.59         2443.60
o.loc[:, "unit_price"] = o.loc[:, "unit_price"].apply(
    lambda s: s[:-7] + s[-6:] if re.match(r"\d+\.\d{3}\.\d{2}", s) else s
)
dfs["orderlines"] = o = o.astype({"unit_price": float})

j = p.join(o.set_index("sku"), on="sku", how="inner", validate="1:m")
jg = j.groupby("sku").agg(
    **{
        "price": ("price", "max"),
        "promo_price": ("promo_price", "max"),
        "unit_price_min": ("unit_price", "min"),
        "unit_price_max": ("unit_price", "max"),
    }
)
jg.loc[:, "unit_price_ratio"] = jg.loc[:, "unit_price_min"] / jg.loc[:, "unit_price_max"]

In [76]:
dfs["orderlines"].info()

<class 'pandas.DataFrame'>
RangeIndex: 293983 entries, 0 to 293982
Data columns (total 6 columns):
 #   Column            Non-Null Count   Dtype        
---  ------            --------------   -----        
 0   id                293983 non-null  int64        
 1   id_order          293983 non-null  int64        
 2   product_quantity  293983 non-null  int64        
 3   sku               293983 non-null  str          
 4   unit_price        293983 non-null  float64      
 5   date              293983 non-null  datetime64[s]
dtypes: datetime64[s](1), float64(1), int64(3), str(1)
memory usage: 13.5 MB


In [ ]:
#jg.loc[jg["unit_price_ratio"] < 1e-2, "unit_price_ratio"].hist(bins=[0, 1e-4, 1e-3, 1e-2], xscale="log")
#ax = pd.Series(pd.cut(jg.loc[:, "unit_price_ratio"], [0, 1e-4, 1e-3, 1e-2, 0.1, 1], labels=["<1e-4", "<1e-3", "<1e-2", "<0.1", "<1"])).hist()
#ax.set_yscale("log")